# 01. Chuẩn bị Dữ liệu (Data Preparation)

Quy trình này nhằm mục đích tải và tiền xử lý các tập dữ liệu, đồng thời thực hiện chia tập dữ liệu huấn luyện và kiểm thử theo thời gian để đảm bảo đánh giá chuẩn xác cho hệ thống gợi ý.

---

### Phân tích Quyết định Thiết kế:
*   **Tại sao chọn phân chia theo thời gian (Time-Based / Leave-One-Out)?**
    *   Trong môi trường thực tế, hệ thống gợi ý luôn nhận dữ liệu lịch sử và phải đưa ra đề xuất cho hành động tiếp theo ở tương lai. Chia tập train/test ngẫu nhiên (Random Split) sẽ gây ra lỗi **rò rỉ dữ liệu (Data Leakage)** khi lấy tương tác ở tương lai để dự đoán quá khứ, làm ảo tưởng hiệu năng thực tế. Phương pháp **Leave-One-Out (LOO)** lấy tương tác cuối cùng của mỗi user làm Test mô phỏng chính xác nhất quy trình vận hành này.
*   **Tại sao không chọn K-Fold Cross Validation ngẫu nhiên?**
    *   K-Fold ngẫu nhiên chia cắt hoàn toàn yếu tố thời gian và phá vỡ cấu trúc chuỗi hành vi của người dùng, dẫn đến kết quả đánh giá không thực tế trong hệ gợi ý.


### Bước 1: Khởi tạo và Tải dữ liệu
Tế bào (cell) này thực hiện các công việc chuẩn bị ban đầu:
1. **Import thư viện**: Tải các thư viện Pandas (thao tác bảng), Numpy (tính toán số học), và Pickle (lưu trữ đối tượng nhị phân).
2. **Thiết lập đường dẫn**: Định nghĩa đường dẫn động tới thư mục dữ liệu cào (`data/crawler/`) và dữ liệu giả lập (`data/simulator/`).
3. **Tải dữ liệu**: Đọc 4 tệp CSV chính bao gồm: người dùng (`sim_users.csv`), lượt đánh giá (`sim_ratings.csv`), hành vi click (`sim_click_events.csv`), và danh mục phim gốc (`movies_crawled.csv`).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import pickle

# Thêm đường dẫn cha để import recsys_utils
sys.path.append(os.path.abspath('..'))
from recsys_utils import split_data_implicit_leave_one_out

# Cấu hình đường dẫn dữ liệu
data_dir = os.path.join("..", "..", "data")
simulator_dir = os.path.join(data_dir, "simulator")

# 1. Load các tệp dữ liệu simulator
users_df = pd.read_csv(os.path.join(simulator_dir, "sim_users.csv"))
ratings_df = pd.read_csv(os.path.join(simulator_dir, "sim_ratings.csv"))
clicks_df = pd.read_csv(os.path.join(simulator_dir, "sim_click_events.csv"))
movies_df = pd.read_csv(os.path.join(data_dir, "crawler", "movies_crawled.csv"))

print(f"Loaded {len(users_df)} users.")
print(f"Loaded {len(ratings_df)} ratings.")
print(f"Loaded {len(clicks_df)} click events.")
print(f"Loaded {len(movies_df)} movies in catalog.")


### Bước 2: Phân chia tập dữ liệu bằng Leave-One-Out (LOO)

#### Nguyên lý chia tập dữ liệu theo thời gian:
Để đánh giá hệ gợi ý một cách chuẩn xác mà không bị rò rỉ dữ liệu (data leakage), ta áp dụng phương pháp **Time-based Leave-One-Out (LOO)**. 
Công thức toán học chia tập dữ liệu cho mỗi người dùng $u$:
$$\text{Train}_u = \{(u, i, t) \mid t < t_u^{\max}\}$$
$$\text{Test}_u = \{(u, i, t) \mid t = t_u^{\max}\}$$
Trong đó $t_u^{\max}$ là mốc thời gian tương tác cuối cùng của người dùng $u$. 

#### So sánh các giải pháp chia tập dữ liệu:
| Giải pháp | Nguyên lý | Ưu điểm | Nhược điểm |
| :--- | :--- | :--- | :--- |
| **Random Split** (Ví dụ: 80/20) | Chia ngẫu nhiên các dòng tương tác | Đơn giản, dễ thực hiện. | **Rò rỉ dữ liệu (Data Leakage)**: Sử dụng hành vi ở tương lai để dự đoán quá khứ. |
| **K-Fold Cross Validation** | Chia dữ liệu thành K phần ngẫu nhiên | Đánh giá ổn định lỗi sai số. | Phá vỡ hoàn toàn yếu tố thời gian và chuỗi hành vi tuần tự của user. |
| **Time-based LOO** (Lựa chọn) | Lấy tương tác cuối làm Test | Mô phỏng chính xác hành vi thực tế (dự đoán hành động tiếp theo). | Nếu user có quá ít tương tác, tập test sẽ không đại diện đủ. |

Tế bào này gọi hàm `split_data_implicit_leave_one_out` để tách dữ liệu. Tiếp theo, ta gộp toàn bộ lịch sử click thực tế của user vào `user_interacted_items`. Điều này rất quan trọng để khi mô hình thực hiện đánh giá (Evaluation), ta sẽ tạo mẫu âm (negative samples) tránh lấy trúng những bộ phim mà user thực sự đã click hoặc xem. Kết quả cuối cùng được lưu lại vào thư mục `processed_data/`.


In [ ]:
# 2. Phân chia tập dữ liệu theo Leave-One-Out (Implicit Feedback)
# Sử dụng ratings làm base và map với clicks để sinh test set LOO
# Để đồng nhất, ta dùng hàm split_data_implicit_leave_one_out trên ratings_df
train_ratings, test_data, user_interacted_items = split_data_implicit_leave_one_out(
    ratings_df, user_col='userId', item_col='movieId', timestamp_col='timestamp', seed=42
)

# 1. Tính mốc thời gian tối đa (test timestamp) của ratings_df cho mỗi user
user_test_timestamp = ratings_df.groupby('userId')['timestamp'].max().to_dict()

# 2. Quy đổi timestamp của clicks_df sang Unix Epoch để lọc sạch clicks tương lai
clicks_df['timestamp_epoch'] = pd.to_datetime(clicks_df['timestamp']).astype('int64') // 10**9
clicks_df['test_ts'] = clicks_df['userId'].map(user_test_timestamp)

# Chỉ giữ các click trước thời điểm test của user đó (hoặc user không có rating test)
train_clicks = clicks_df[clicks_df['test_ts'].isna() | (clicks_df['timestamp_epoch'] < clicks_df['test_ts'])].copy()
train_clicks.drop(columns=['timestamp_epoch', 'test_ts'], inplace=True, errors='ignore')

# Cập nhật user_interacted_items dựa trên train_clicks đã lọc để tránh rò rỉ khi tạo mẫu âm
click_interacted = train_clicks.groupby('userId')['movieId'].apply(set).to_dict()
for u, clicked_set in click_interacted.items():
    if u in user_interacted_items:
        user_interacted_items[u] = user_interacted_items[u].union(clicked_set)
    else:
        user_interacted_items[u] = clicked_set

# Lưu lại các tập dữ liệu đã chia để các notebook sau sử dụng
os.makedirs("processed_data", exist_ok=True)
train_ratings.to_csv("processed_data/train_ratings.csv", index=False)
train_clicks.to_csv("processed_data/train_clicks.csv", index=False)

with open("processed_data/test_data.pkl", "wb") as f:
    pickle.dump(test_data, f)
    
with open("processed_data/user_interacted_items.pkl", "wb") as f:
    pickle.dump(user_interacted_items, f)

with open("processed_data/user_test_timestamp.pkl", "wb") as f:
    pickle.dump(user_test_timestamp, f)

print(f"Chia dữ liệu thành công! Train ratings: {len(train_ratings)} | Train clicks: {len(train_clicks)} | Test instances (users): {len(test_data)}")
